In [41]:
import pandas as pd


df = pd.read_csv("spam.csv", encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label'] = df['label'].map({
    'ham' : 0,
    'spam' : 1
})

print((df['label']==0).sum())
print((df['label']==1).sum())

4825
747


In [42]:
from sklearn.model_selection import train_test_split

X = df['message']
y = df['label']

# Split training data and evaluation data. Ratio 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.2,
    stratify=y
)

In [43]:
from sklearn.feature_extraction.text import CountVectorizer

sample_msg = df['message'].iloc[:5]

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(sample_msg)


print("Original Message")
for i, msg in enumerate(sample_msg):
    print(f"{i} : {msg}")
    
print("\nVocabulary")
print(vectorizer.get_feature_names_out())

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"msg_{i}" for i in range(len(sample_msg))]
)

print("\nBag-of-Words matrix :")
print(bow_df)


Original Message
0 : Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
1 : Ok lar... Joking wif u oni...
2 : Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3 : U dun say so early hor... U c already then say...
4 : Nah I don't think he goes to usf, he lives around here though

Vocabulary
['08452810075over18' '2005' '21st' '87121' 'already' 'amore' 'apply'
 'around' 'available' 'buffet' 'bugis' 'cine' 'comp' 'crazy' 'cup' 'don'
 'dun' 'early' 'entry' 'fa' 'final' 'free' 'go' 'goes' 'got' 'great' 'he'
 'here' 'hor' 'in' 'joking' 'jurong' 'la' 'lar' 'lives' 'may' 'nah' 'ok'
 'oni' 'only' 'point' 'question' 'rate' 'receive' 'say' 'so' 'std' 'text'
 'then' 'there' 'think' 'though' 'tkts' 'to' 'txt' 'until' 'usf' 'wat'
 'wif' 'win' 'wkly' 'world']

Bag-of-Words matrix :
       08452810075over18  2005  21st  87121  ...  wif  

In [48]:
# Model's pipeline

from imblearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from imblearn.over_sampling import SMOTE


pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('smote', SMOTE(random_state=42)),
    ('model', MultinomialNB())
])

In [ ]:
from sklearn.model_selection import GridSearchCV


param_grid = {
    'model__alpha' : [0.1, 0.5, 1, 2]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [ ]:
# Train model

grid_search.fit(X_test, y_test)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...inomialNB())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.1, 0.5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information printed during fittin

In [ ]:
# Predictions

predictions = pipeline.predict(X_test)
print(predictions)

[0 0 0 ... 0 0 0]


In [50]:
best_pipeline = grid_search.best_estimator_

print(best_pipeline)

Pipeline(steps=[('vectorizer', CountVectorizer()),
                ('smote', SMOTE(random_state=42)),
                ('model', MultinomialNB(alpha=0.1))])


In [ ]:
gs_report_df = pd.DataFrame(grid_search.cv_results_)
gs_report = gs_report_df[['param_model__alpha', 'mean_test_score', 'std_test_score']]
gs_report = gs_report.sort_values(by='mean_test_score')

print(gs_report)

   param_model__alpha  mean_test_score  std_test_score
3                 2.0         0.963229        0.007175
2                 1.0         0.964126        0.004912
1                 0.5         0.966816        0.009659
0                 0.1         0.969507        0.008695


In [47]:
# CLASSIFICATION REPORT (METRICS & CONFUSION MATRIX)

from sklearn.metrics import classification_report, confusion_matrix


print(classification_report(y_test, predictions))

print(f"\n\nConfusion Matrix\n{confusion_matrix(y_test, predictions)}")

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       966
           1       0.94      0.97      0.95       149

    accuracy                           0.99      1115
   macro avg       0.97      0.98      0.97      1115
weighted avg       0.99      0.99      0.99      1115



Confusion Matrix
[[956  10]
 [  4 145]]
